# 🧩 MISSING PIECE: The Epistemic Caution AI Benchmark
### *"The answer isn't always there. Does your AI know that?"*

**A Competition-Ready Benchmark for the DEV × Kaggle Benchmarking Challenge**  
**Author:** Missing Piece Research Team  
**Date:** October 2026 | **Domain:** AI Evaluation, Robustness, Epistemic Caution  
**Tags:** `devchallenge`, `kagglechallenge`, `ai`, `machinelearning`, `nlp`

---

## Executive Abstract
Standard AI evaluations evaluate *problem-solving*: given a premise, can the model produce the correct answer? This setup creates a profound structural bias: models are conditioned to presume every problem is completely specified, well-posed, and answerable.

In production AI systems, this assumption fails catastrophically. Real-world tasks frequently present **missing numerical quantities**, **omitted relational constraints**, or **mutually contradictory facts**. When an LLM faces an underdetermined problem, a reliable system must exhibit **epistemic caution**—recognizing that available evidence is insufficient and abstaining rather than guessing.

**MISSING PIECE** introduces a strictly controlled, deterministically evaluated benchmark of 60 base word problems generating 120 paired examples across 6 domains. Every problem is paired 1:1 with an intact twin, isolating the exact effect of information defects. All scoring is deterministic with zero judge variance, accompanied by Wilson 95% score confidence intervals and paired McNemar tests.

## 1. Research Question & Central Hypothesis

### The Core Question
> **When a problem becomes unanswerable because a necessary piece of information is removed or the supplied facts conflict, can AI models recognize the problem and abstain rather than produce a confident but unsupported answer?**

### The Central Hypothesis
> **The Reliability Gap Hypothesis:** Performance on answerable problems does *not* reliably predict a model's ability to recognize unanswerable problems. In fact, high associative fluency on solvable tasks may induce stronger answering bias, exacerbating confident hallucinations on defective inputs.

### Why This Matters for Developers
If an AI customer service agent, medical assistant, or code generator receives an incomplete specification, returning a confident, made-up answer is far more dangerous than explicitly admitting: `"I don't have enough information to determine this."`

## 2. Related Work & Novel Contribution

Recent literature reveals an urgent need for epistemic evaluation:
1. **GSM-IC (Shi et al., ICML 2023):** Showed that adding irrelevant context causes LLMs to drop 20–30% in accuracy.
2. **GSM-Symbolic (Mirzadeh et al., Apple, 2024):** Demonstrated that symbolic and numerical variation causes frontier LLMs to fail due to pattern fragility rather than genuine reasoning.
3. **AbstentionBench (2025):** Evaluated broad QA abstention, uncovering that **reasoning fine-tuning often degrades abstention rates**—models become overconfident "problem-solvers at all costs."

### MISSING PIECE Contribution
* **1:1 Paired Twin Design:** Every defective test case is paired with an intact twin sharing the identical story, names, and numbers.
* **Tripartite Defect Taxonomy:** Systematically compares *Missing Quantity* vs *Missing Relationship* vs *Contradictory Facts*.
* **Deterministic Strict Scoring:** Evaluated via exact JSON schema contract (`ANSWER` vs `INSUFFICIENT`) with mathematical ground truth. Zero LLM-as-a-judge bias.
* **Statistical Rigor:** Wilson 95% confidence intervals and McNemar paired discordance tests.

In [ ]:
# Setup and imports
import os
import sys
import json
import re
import math
import base64
import gzip
from typing import List, Dict, Any, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Environment initialized successfully.")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

In [ ]:
# Self-Contained Benchmark Dataset Loader
# Unpacks embedded benchmark dataset (all 120 validated paired test cases)
DATASET_B64 = "H4sIAAAAAAAC/91d+2/kRnL+PUD+B2Kxge8A2cfnSLKRBOtdG+fD+XG36zi+XwQOSWloccg5PiQrh/zv+aqqm+SMqFGPtoe+DJCctZzmq7tYXY+vvvrHq+zXeL0psqs8ffX5q2XcZFeu516tqzS/zrP01Zkc29TVssjW26Pw4ybOaznIf8nBJG6zm6p+kKF5chXXebtaZ22e0K9VmeZtXpX4eZ03TV7eXP29i8s2bx/oiupObfZrixFvnGXVtfnfu8xZxakTO2nebIr4wamunSaJ67us+cz5sMoenKYqUseL9FFn+eDE121Wl1VVnjlxSScnXdNW66x26qzt6jJLndDZ1NldXnVN8UC3ulm1wxXi5NZpKycv77KyxQt95vyxunfWcfnQj4nrzKnK/qnK6v4/8RLZr5ssabP0qmnjtmvwHt989/7Hr7/+5u03X333YTwgLpv7rH71edkVxVk/H3l5XdXrmGbpqn3YZKOpyktMXlzgylVyiythna7zpCtamu4MM4dHjWuayZuszGq5RpNhKT8P/bNXd3GRp+qgfra7rNaLPfq5rNqMn1xuOMzCaOqdvHGqdd7iTT579b//+i//mBIn/xIP3cZJu0eY/MsJYeKDI2HCy2SYsPSqzdfZjiT1d3gkP03bpXhwLHmcYr1aiEHT4u8HZxMnGb2MH+HPG7zNBpKRxuNlbqsWry6/phX+p11lW1d0Ksyec0mnNdMr/+a79z999depNX/l+9GrfYsuQrG1wnjPrF5naY65sLfGP5b8haUZXRwS1uBTdW7qqsNX09Zdu3Lovjd4U3xlHZ369HIvjLTHYkp7LHa0R50V/JwQd8x1U5WYpp11x99tHad5wv9+tPzfxg9Y7qSq8amf41uv8B7NmXOfl3QtXk3MeLJibeEUvKDXjnepRsrqNhnd0cEHnmSsbNSRT/kIHX+gUeou+tyRFKmLpXkq8vPU2UdSHcMcYVqv+GGu1nF9k5dz6I/3E2+bxCV+hka/w/zjxfUMtau4HC0KLVNW79EsRqLmT4mavyNqvXa7wlQlt48F7fmt6ifsBStsJJnzxllhM2ogRy7pmqxw6iqVfcopqhv+vBoSjZg+KdywbK6zmqQnxkzUdEdWsv2ptAsNl/9yvA8NY+psHeclvtTR0DdH3o30s1/FayiLdg55+jCaLr0YRvuQGzy/D7nBhKzwQXOj5umtCNtOssKKKzsjpB3otqGlxud4m9WsmaBsPU8foNWkk3jkZ867ria1FdetkxRxg/EX/ci0xpQ53YZNHaxiRkYJtukUczLIy31Vk3D3Z4nQHLxzBYtDN65jiILNbcuLnpcOL5oyeaMDNckeU2VVwSDp2iKDEYqpwpQ77zETNawLJRkLEpq8ZqFpyiymRezlgjQnhp45C0+NYjnYdHWywgukLBqRO/5tWYssQm6u62rNl2C7F/ZlDfU03sV278vGby4KWz01tlH6V1byPqqf/WDx8qLg9Ayj0MAODqfs4HDHDm4yPFOpvIBs05jKV+tcfO66zptveY3abL2hKeqwiPdx45z7eM2bOiOH6kvyZMht2h1YV9hUsMieOwz+aQWjmq6wOxiH6SoHr/6Ff4KLb6Bewin1EkZWFv/DfeUk5ECRLXGTtbQN3Wf5zcoJz53bG7FNVll8l5N5RgPl58YJQv69VwO3N4MnVGBEu33C4d/6CX7qnsGn7k196t6ltc1kWVW3pMTZ32CTEnuAE0X4T50m7NiU+N8CRsTiAsodny0rfn2Q9gpv4cRdmld0LeftO1yp1R98rr53dpATMv7YjqFVcXJogUZ2BgxTt3+BYFyeoB5YGOiBxZQeWNjZBMRwrCvs9Rjf4GV4T/fOITAPjaz6hXOT14W2RsvKqVhfqLhHs+V+yKGxKTDcgA6QfBxuXkanZl66nomr6npT7odnPabaJKuqKqC+lzVmDE5AkuV3sA8D37mWUIqoDxU2TbMCP0vkr6zKT7fH4I9vcX+Km31fOh+6rGEbtBcNuDzJLS4OE9HxQnWSFo2RLJXZvfpx8GRlk+GnnCmuOpq247uyeLuyWy8hQZMzC+XZlQ3eiK/xhZoXsvTld4RRnCXsLEyelsT/+Hes16dO8rQkRv7zGijyJ+SQDx4WnXtKC33Z5UVK7/GG3tFb+Ng4WvIoIBEQif7nL+nnaPwjxUsoSDRcQSSIfqPB41NLdeLhO8/i/PR2nuDCRAMFFxMrzwe3Vr6p4FMiGF8UVRKr0OuUEtIy0qzyzaSRQo4krds65rDEkmPqKfmkTkdXQJAk6Ta8RV0XuKkTFxWO8sZ0H5PtSSrqAfLXjlSJPkVG9Oaq3K3EghxZmUCh5pvsiiXg1SyS8lf65/DKiBfKdIkKwTX2RcYiI9GIpkQjurAdsqcwSE4BropMk9fB+WfQEr/AMs36YzBLOWrch4v5DDFZ6Z9qdM57Dxm2OyNHkpJWRRHXo+TO6E7zhOPrHB+SnrZ5hOVP6h0lCk+vOppPmf37VV7gg8noE9yaRJnAfXbO80FWC1bOniDaIbYN8sX/ZAbNPqs4ODWr2Deyiv0pefE98+Sw6W4EN7am+EdawweWeOj7ls9w/ptUqv7Hz1tpZBhoGYc7Q9dZ46uRPPKKtC+WCBsY/SnZH+UgPX2bn8e3+RvdJox2LjrlfyPS38Zlwi6YjMbjSCC2zenDrvPNsTNBmPartKtnVGPv1N14movsRmbvb2xOyFN9McyMUnbLQXL37YjBpZGxNBXCCS6PZSw13XrNydP1BgtbY+eC3EUBHyD7mKaBFr3bJEiGQXFDQaVVRQls2onH9hG+1iYrKhXDrxFLyCmfpq7QZPqaRxYaXPH6etpMOpa/hfdPyRpSb/hpW33az4bD9zBKJkYGcZxoKo4TLax5UT9gG0eKBuaRCtVGhD2AHoay0b+xB6UP9/Fa3sn707eQCjyOraHhEi8L7Ebu6TlRkWuw7FN4g8i1Er77/j5TcVxotjUW6XdI6FJe0HUStk0oWfx7Nm1Cp8xhphT9mGg8ZGoXwdx1DIiiu3wCOwegDDZs+LyD1//8BNc/NDJYwimDJfQOkoBnvaQ+KHeHbxl5fk7/45GTFkjGwGXEAOPWtGVKm2OLNFBJX/OyqmEmp6MYbz9eXZjTxQ/3K8oSQ4qQCf6mHf22c4d31T3yFPcli17gbf3244Z+2cok61spRBS9grr0M0Fjy94X3+kKG+s8FssHXgdxx+nbC9wzZwkvAg9A3x2MlV8ICLW1RJitmr7aXxPcuIE9+AVBVvGGGX2h+mX2eGSRUeR5KvfoRtYjz0l1fZ2RG15tBLELWeFojfKN4IdygJPEyHflJ+ipdZ6yP8ZI37hoKjlZXS0d4A8Z8mX0N2yoa7L2rsmvTYqKHmw3Vc2LMcRf+V582ZaGPYlbsGb9xClN3V12RaI2h/3zlZocvKlK2Q2m8h5b2MDUCaZMnWBxqCW8BzqFGARZLrT3XbiwWJusJq++pT2uFdu32sjrlWTlfOZ8pXShx446FjhjdRk+PhmzEDv/9c0P6sBIUw0HtaUspx4O9XVPDjBlAKfzpuB0XmArx10633zAwmLOshE8nUQEkO4ixrazwQbnBXDRYwaZ3pP7hPz3DWPtSMdcYv+DCZ419CEQvHdPcls8ozYjOB6meYOtEaqIL6NuhrHrJXmWB8tHeIoZh3MjJ/p8SnWcH8uJxgpt8NykBQtUgyiVgjAef9iEiUjFasnxySM4u6YgovqtZhDmTnHA2x9+dAi8vZ0D373LkbeSZNNh28XzXNVdkc1jzLzlrZU/jo2aODPX2QAcE01FVqJLa64zVAcBW6AvcpheLAFAxiFOwjD9uzwl+6ykWJzCzrzhz1z/Cwjs4Qc6e6FOHkMj+GJKJvoTD1cMJ+hFeUaKwZtSDN65ddz+GzbIMZqsA0FM9UkDLCIF2ZbVrxJTpdmQkD4h/On7vhwdI8/6DDUOXUMQ7qIisGT/69TWMk7+64voex9ZY+CVrpIpzP5xxOe7/k23ptSofszA0PCnDA0/sFE/BrkipV6jVqXIVAWihJSpdCzQMX/KCECpYBYoZibD2RHR2YHAfRTI73UFZwh6b1juiBP509IXqkr9I2oGDg/GnGAsbmGUtbYUhH0mHmMrFitW6U7otZGqMrnay2O1lmMnfKf2ikwdhASaubI+w6sOKZ3puVRJbH5Ornnpc9xD6Ptp49WotiyYivUGru3U5B9zrrdYZqjZU5oGyugu1pViF+eBg0BIqwMkTZIpKASXZSkFlhfsJgc6Icn70XAdDKJrQKD4zFWuCoSwqR15L+qf4Sop8vWSEkJz2bB0P5INBrDrGejzt2boGd9ID1mC1D2jh2wi60QXbZ+iA8xetHPll4HwLKuklaikedE02xM0nkrM4APECXDojGu54E8sOfZZ98pqjwZyjXLQ7pSn5F5aDty+XRUoTVN1iwvYJnlN5utg276HxsC9U0knxM5rlJkt86KYgFpRQgmvfsMz0SHcKhdXdu+RVU0HmO8VIZ5m2qwI3aXcY5krI5M3CI02n3Bq8wmPFTm5o+84LyFFkFukICim1nDFR4KQF28tZAGXCJkUw6Z7vqCKoThBiIQ/EnwDZD6nKEKRczH4pssp+1NmY/NmfaMLSvjySlY4WzTxIEcWm7RqYC9AcOBu1s18CBe+raNXhgXpNkcJeR1zwkhMMKMovme0S3lTu5TnH8HnJi+qzpedVCZ1S47eZ7LA91l2qyuhiZsDGlWqVLhKSRk5vY8ek+miEC8kuxQ1Z4cr7E+lfYx+2rBLTz5a0f/GAknxu60E5fYtNYAP9hKbRt3yyPKGjAXwu/ezVyPoSZIJGJJEzzvpRhBifwpC7F/YNpffV9eoQqNSd7yKf67cay5GY9ESC5qOwpK55ypn2r84rziIFe9hNJ5s6RivD6VFbv+yKrtx2Svf7ZNGhXSymBOYvM3pHU8Ds+hiRxYceakreqk5ZOaPwxwqE9pAFbkGAWAbZs1TMZ1njBmHIdm05M7rC47pHd282Zc4PL3AjR8aBPSmbBs/tBHQUxUScXmraFqCy3NEEFqh2+AqsOKBig42qOFwkKRvdKUJ0LgXeqT2Ec9UlAFxQQrb0YeA6bujBV9XRJHVn+tf7p47kh71S1/AMcol8ZMylRkcNlZlhycRLk+xpD4wwmJNRYbDwC4W65e4ZixOTLlfSgYha4gR2CXeQifEJEQ4jhpbdZyhWFxsIvVLIisp1SJgL3rDa03oEAG1rM/6weCX0zcafHF/dL/BfFZ3GgmSusAsnvhgI1KVMdKnMzrkb2TmRnOtPmtlz6i44UOeFbTNN2vlsvP+TQqe8CK8JLAIvmDjkgul8UJ7NjUDpeZOKTU3tMUuVD1QaUrTDhLiXziE1VvC87vta7uXhPKUQ5BPKK+0psIE8k1VRcvoHApZUHwD89QSfOJR+ltfm3gG2pziPyoDTk9yeImLf3IlLkZZCn8qS+Ev7Je4oGSFPceIo1LrZSGUULT30H/FVlXbjE45YbW7awidoOzIT38kBZQEgO5iljtdSce3Unc5tn8u70BuE2AXM8f/3uiJZAOY9nZ5mq0w8rNGcWSUb7CBLX9mN+sh5v32ooHjoxK3XKPDlWoUkfrngKRb3snoloIT/g0wwgJlk0kaHkvmgYulb1MCXcMEBSgNKGJW2JjEggqiieyFT91jQxkhMcIpJEZ4btWG0hktqB+kueCeU11XQdVLl4w8FwOJYkYslN3mniKIkYxp+m37piIdRZMi2Qs47cKK11/unKs6ZcbkiJK4/v5z2UjrCmuOHE8yEeA5jmx9W93R9KjJU3YQQl+4KX1UYEBliKRMi4BT9kQUDXDB3tS+5i2skRzRtkSYT/yQduy0o8ifqEMJm0EYHI6PLrtiCVvnLx1mpH2QDwklwjmV8/JsOQGUV3ZNIC14bjxcYoYZB/xoG8Pg0a533ZWJJOrUaEWelzfq/oqaQ1U6SGVdExeH+//e+clRLUZGFQnRVEVCFNlOmL6FU+SpUlwoznP4UhsEfOiwrw4jBo1jkLWmT9vxWdr157HYnBjqU3G6Ex8V0bxypYzeRxfq0iREdC0OHur75PxhUrZ0Mw96gx8WmfZSfW9zJNv5XQfUxqMJVfOpZ4OLrEUj8dPucb4MVJE7pYrchRXn6+u4psrcP1Ur5GhyJgeA0MSbjfjk0Aq6okf/7DstG0txIwpFj/2gyPnY5ScyzRokAvE9+D/PQb5IQejhsloZ3VCVkBxV4I0HoVZ+Of/WxUkyMJqYOhY4GJ/ROB9JxTjQnEywL6I+7NICXaNtXdOt2W8liYFKRBJlHoXz5633VYpHpnrAgm3N9BmMP6Yn55I8YSFXpri4Pk+roAsDFTSV/nIvrKigb8h8I+B65dzEa7zIn3O437vU96QS8D4IUmP3Ao0O57Ie/eaLjF08+iEYpb3o+n3Wi++DLD/4YaHTEAUixZXJWYeHqy8vTpBd2iT7boVf2gjxPgfTtNQ3spG8h3F6JrZpi9xe1DpmTuw87Qdw0qjhiyM8+48mUc3fDmPgns3QgMcgnIo1ha7Fmk7B1HPqBDMARROhCOfH96NUaivaDFoGWfgeZF8ncapKjxQchMqjJemqs9B99EqfS9E4b7j62H4a3WGIXuqHe2niFgVFJ1i6ExjBiIIpGFHgHweZ9i5uY+dtRlPnfEV+VcUM5FhHP8CWxC69FPCR/hoN/ikbD14LVTKPE8uALwYJxN8p3BKiTCZQDokGrdlIgrbvIkL01J2OTanDzzAn9SCrJ6EfrJ3BIXcGaw/OXgvsi0pnw/xNaW4SnhuzGqDICBAZTeXXotB2yOADJTb+u7etor7hD//ws/7B0z8AVH6v/Vw5V+wrNZz6mgng/oazd2JL5aNGONOthMZ3m8eMx5242PTJio+jwNLkNcctg9QEy7SOWweplZEwAoZqU2VraveGEUwQ1xYCCc/baIeHFBAr4cp2Ini/yyQn8xuEFaxpsjWEKt8U8M3m02N6TuHY8Jvr4v/aGZ7G3Mgy2ivDqb0y9C0nV8hrICMR7i6S+YhyNiuGb/eAJ9ZIZXYnVgBz/+gxbHUp1bNNidn2jRd9X43mAT9laTkeMvgF/mLsFuhThqJpuWNvhQ2PTTeZKzFDTUxgTxdXwKzPJX0/CKePRBua4cU/mVyTT5SOa3KSy7jMpIGlVJqQ1vuET1DTWfVL9skeF9UggOFNBTC8C2vpHBilCVVhx3lTSMNPSmHCPYXtx+qsWFMnPtTR3orZjzulmKQRLdU5HKKsWemzhmhZAmP+FpLIrImX8A82THmor953P2rwhVyPuRInb8/eKKGx4jvU1cXLItNoCT7/8J43J8ft6hkwAXtTRGmeZ02e7vtGeD2sG7CEe2xs2PBUDJYw/hXckIIA/sQBpZuwXQgj1EbHBpGGZq2vKEd5TigYtpBxHNxI6Ak54VyRn4pExhYjsDzpzv3kzP1l/3sThOEJBuvPDeITFlAJe/hBTOAICL/sQSC8HGiwb7lPcLUjg63HFvv8vu57L6Kd/ygy+X30qcFJQraf/6wtALafXmS7SO0Xo633MXWcIFWHb6DO/Sl17p9baQF+nwtzdkUPRrkORAZB2dYI6GmDfgGgC6Q2qzgoSOi+zt13Vb/mZpI+kp6DqVpauYtCcahTDq/2OV+cYNDYN2h+6E/lvvzIxvLfZdiDYaBfd2iVPKr7WSxGZTc32C4E2ix+hZxC8NaOcBqeDB2PVHySns8lucJyMFnR058Q1/0VUUlG7QA01MW79EZXeUG/1BM0CoyolqMpDyLyrDelKZhjkisq0GyI1rVAW+UvqyUdXAzHel4HOkGcz/7c0c84UUSFvHRizKALlXKZWUIcVM3O0Y3Z0F/0igP2qwfy7s6ZCmz0k7YT4diZw33S87xJaUl2ntQ8LxKaA+ViH4rrBF2FwKB22FJbjj32xP5GHIQexS+AgHYqb6m7TmAbGZpNECRYbSHucKqtHh37PItTCzWFRuQD4ZQHGV7YrQRtVl3bUuesrtG14qLqsKi/QpWtYRcg2cntgqjrLgK2JbERIxJUI1dKGlG4/BvqykscoszGoId95nxXjc+SsjsIF9Obj8AzWyP6yOTwbHOF0pMYEHgkuK74tefZab5PEFYDVcNDT5o/uQQ6YKPn7wtgjW+Ij6Z42Kr5dJoH/He9lyrueZ30sURxT2mjl1HDbbNVMmaGKyb8HQI0q3xxew1Y9wSbPQdG+WRL3PnGFaCHMumf6SI/ZiG9BmYUkyCbm8RERiT5L6bOt9iHjG45miZUOhSPNc8xtrQf6gqRff3hjd/fBOISGhSQh1MAlzC0FBtbMtFEIinvZVww2w2FO15fXFI0lBQmSC7uEfbA+72GVUs/bihn11Bud7tbODaBsol5o9zhPlZb3fT9Btqcg5XIwjs9FeIamTY2gOXPQ1IOhpgL48CTKHMoC0W6pK7xCG7eN3CRwJ2G89RdaRGIbrOfC/bWOTTNezU9asJIvT7PU+oZ8aXYadlhgEE3ad4hAvRk/46hFTSMGVRkMiAY1BU5K+AZG3vYQ292y6ueS2FOjPm3nCXv+d76yfzD8+2BjMQqmBKrIDiWkXMN9Jy0dfA90gaS5+8aBmpgo0klzEt/0evqlg8apwr1oYQPnhTg/tkW1lf3+lBPKcSuauCR5UPZF/PZNAO4l+eIO5rsTo4ZQaARgvejab9MBWQmEjByhvOyk9jlFiOYXe4ve0Sn9PwzEnCTfPVzplgv+xqOZ4XKNdI97pTucQPrve1acn2xUopZMOTqf15XpIe3yqdiAhoUCNKcochO/Ui5IIGU0KfFmCBwpCIYBMU0EhbillHsVsrfJmbSI0sFGa+bqyb/n1lIJb9Vr0ZfF03hmIb9C7V3a0YvPQ085LYk1M0+PO7zHpYFNO4eUNETMNx/TlTtPpBCeJIgBZMd6mNhCsbdvp8HLejGl6R0dONMbh7DVMgwfel9baEW7HaKH5i0ERkub2fabOi1dY96Vi+E9zDqgWkAYLHUyO7p+IxR6zoFZBkd4mpIJI+Y2Jm9pYuhaZ3Vdnb7ArzeKRLZGhV4+FNbiu/b1xjgLS55kcCJ3zQDJ9KmaqRRR0XV3UpnEKSVCxK5+0lXCoObivR/Q70P+SxHKPV6kwzH4vRBB1i4y+5jGdI3lJpweqqjO87ysvwLTyX1eZircEPfeSAA1Kx/ZMBVZcnv9AeDCI2BgWKHof9pJfOb8fJTKi5cHIWWf2+V9skVXrgGe5U7tVe555aYaJfVfTGQ0KIkJ6EMK9chcg6SXRuJuGBjkuqzNcXyQpTh5BljLb8vMxWa08ccsof448V4bq4ZqhF5PdxhVLfDR7TVCxmR60gNY5Fdt8cHTvw/SFEaQDODKWhmENkzbCgMB0+X2+Rwvh4f/zUVshXwOLmu9Z7aSMWNbF0+tWqF3ECMNOqSwghUsJnLnYeOMo6u02zISaK+Kaw++qsLS+v4auMK2InhXPeFh+QGsS+k14p89zRRvibW0MfifM3bFb0Y9ZtzMAZbqDQ1cn6BZJdQXDhhG6h7TLivvd7Q/Aib3jaaxyz6mqY9o9tA6SYPfxg9xWAn9XNg1hXLSMAsaStTMftY3QURwyPcS0hP82zjYimyVzp5bkchWROo4ebIQwDvk84oVm9GSr4Xo/GEmvVsDI3Yti0xIBkjcyzxIdkgNrKnfRjskTxcaYammZML28RQXOWtp5NKu2RyjChoDGrXoqlMQxRYbVsv/fWoyrwVTxOBPicI1A8U/9GlS+diaVc3oyp48ZgkAMDjZZfabl2fjDrX00nb9zzcx1qcIC7HKPTjTjnrrm8584R8SdI1PY8fO+ivFxGtHrc8Y6+ZwTbUnAwjXqMn2lr2HNqeBvEodVAYuzJGsM1TKlaGbe9qq9RVPcAzjpW9HGWmQz3ChjdXHlxx78kUjedllH36YvSF6Tkaql46lkB8m1oG9ymchQGseDG1Qy1cawrn2/ghfoTwIoKEstRgLCFJBRVjTF0HUjGSNTKpYu4STis4myJOsmkqK0an86hPeRSNfaAzn+a22udeRScY0jHRNhaCOiZ57heHd2J+/w0bsL95nMeeOsK7NhWaz8AcpkTrPLbNd9ydYTSVnOXVH6WeFFOqKt+o17Q/VVjlXx4h99l21JuBignwYlIkxSiYsiJwNSmVmKF6FeubS+KinUhDEOZzMHzHF5VryHlH54uNU85QxHPRXv9AGpRj5PT+/LrkJVHyxxCIbuIbWYCiP1tNdTxE+hltVj0/oL40YI/sLKjmn5cWcevWKdWXlNrpxWIu95tyezmTnY7mXhk4NKPDXA6tevpJ3VMg8HROzKAswpuSRS+0lhNj6uW6qgRSqhDlcIbuFH0U9kHuKthwo9gCBpAGFYZ8+Ewd9tVhYFLlMJnd8lOgf3LVhcZE2OrqupZvBGtXD7EfnbwPx3NyppJnpL3sSIwR07pl2WHXbDhrLhmyyE9MM3jFLHmzFGUNr47X1nNpFG420DzBlBwFob3U2B19E0DH1jkCfQi2bhjJR3o9oRQD5yf6oBC3tqA97XzRNxLp68bTquFaT2V6MS+Bs+aKvv5ywpF+q8r8gPPgK44prnZOUC2uW06ePH7SF9SBniBMKDIQJFvU1U9J0sdzVr+UhHpfwZ57artPYMBkF0wV6wUX9nTGkvplMKwxZqj4kl2AlFDpKIHJGWfBxJPkPBbch1w0RUK1DkphEOVRIr66VCUMdBX+7tmDL66Oyxm95yUPREmqwwmxTpAEz6gqPJyqCg8Xdv0rrkHojZNY+c1CaaLhWj0TWqQ9aAYI6qJwXm+kGpbVQ99h+iavC02j2pv7w8kDLl0d2oKa9o905GzmLk9FDS+d+0POEk6emGhM8LjFK4B4v5CeVTEz6kclXKJ5M3R49Rd7YjoGusif0kX+xcfTUryvrvNYGiEDuH6uGkpzg0xGEUrmko5SB2ou/iEHPuzhk2wOS3XwgCh0Xkcuny8cFmDnSNnX5ALjEdKd797XBFPWA0LQjHx0zaLDFzt83zrBBlUGoEFvKr7snVt0s/n9aIth6Oew8JfEhEMkS8vqV/GbaA6WIGqEmqGqK/Z8aGe7HP+yAVjdoS6wVdVkww9TtcBlH07th72ckvnyFI1YowhxNBUhji6t0/HB5ekwGP1IOlXnS7gaThb0cSc4IC1fb1TY+17Rd78Rt1n968vPRj/Q1Rb6Yluj+CcQ0vNP420sG+XL+9Hz0iv1LztPHFC9ZcPbEnAUwIOfjXYvxri57qc0M30wXj3gniyXARbVnUJ3uZEl4HICPgIOiVQbKcALPLFpdTceKbI64xALHcfrrfOUS/IU18V4OMzlkom5JkqwBoXDZ/DdOHv1ZAe7fRvS+cllPA1KIWygK55OfNvGVLxGoZY9MMW+mO7pGSdGJDh2mpOYxHRna1PCwd7folOJxeS4KmG/4kD4HL7VV1vzpThN1IRKs0wOAhvhRCMDsziaMoujc5udCYj2o6Hay6XYIKCtz2j7GKH9gohBMyAhADRHDBfw6FB1L8Zw2EZOESXTO5AV4dSpeyFEfRzmlcsTtx/Tvakh+Qua+XqnF+QzQIJa4rt5WiZsM9xIVXnAxpxifPldwFZbowaP+Gr5uD7z9x9JjrOPwdg9Qara5zkoLBDV7mtz8oihVvOiam/ChJ3W97Zoad9ICWaJuzgyTXFxRrXpW02g++stK2wKZ85tlm16DIZ6HuqKpJ7DDqPtPvk6Rd58I04cW0h1A0f9GcB6H+fdg1wPgx6mrsLKChMm115mJERZznHmmM5zuG4L4z8S1W4ftdOAnHSKSOc4gvUWG73ctOdCVjyJajnUJGyqjeIq36O6DCj1wylK/dCzpLqU/fwAcq1SKaKUGYIY6EblHOgtrFt4rtDlTTIVI7PbHYZDCt4BsN3i/1nGAm/rtx839MvZmHqSMBPqWvlOXbo6T2OZ6fnUyJdXiZ4inalvIEP+lAz5no1OMPAFaurgmtbAf4oz8F6CzsiNQ370P37m/IQmu4D/njHONHTJo8qabX7sULNmfFhlmi7lydv8PL7N35hNLpq66Dbp9laomhgaNORQzlT4afqmqD0ZGCwPz7NenuJWaBBQjKYCilFkzXt7GxOkRiQBKwnKr/VmRR0iauTR5TASfzgGi54JVxhywWeJhqHDHDvUp0gJMS3+5nCi9YtT5DAwUCrBlFIJPKvkTGsinXkQ21Yl1F+j3telst2yJ+Dv/SKdWA/df5MvGF/5skvRqFtcJoSBqJIIDj197RPhwgk3a/ucw6XjFHkJAqNGUpYExLTC4XBxIcq3nlz+GWEZlXs2POChojnDU3AFuT1ZssedPfD1b3oa/ZnKJTKsaMmM2tSVR6Hq98+GEYDUMxI8Oz2QTcLWL+qG/Ft2PbZXhM7mog4rzF2gtd1G2qwGxyjbHk5l28NLqxix1rn4HDrpzbdigWZrJo2jgCHZJOcwSLIbwMghJ1+q0KIeTzqI67V2T6xKIkMmSMaSe1+pK4irD9Hik3IxeiAg1KYOx5GVH91sO6M6urg6fxZ3fnTfK6bCm0ey/poziHP81qoCR1H+USkuc+Wr2jOxKTmXuUdbGYmcNyVy3uURtBVjcBgG1Cjto+AXKHNMk0rYLcFk0ejMOyFCKjrLefsO57Q7HQKw5SIFlOgdlAnklsQuLFfZ2yCAlxAima1Vbp+YEOVaczQE0DM7U/Wgnpi+G8Dz5O4LA7d+CtDqL2y49S2B0Jm/kqLE62UhNNlMQYn/SuqiJ7dkflT5XY8ntLPr9I0XmhEKcaemdDRmYFSh2+srHe55R6docxvAN4Ip+Ebg23LJ3sVt7LzNaKqw5JTqZG1JqYtA8pmZYsklXvbR4J+y8WDtpflMmtzQPqRAX3wq/ZuuvpUDG19bxOSp6x8sLeEJcqn4BkQX/hTRhe9awp7+1FvGb1QNnXfpUvgPHGY1h5E5l4r2gPSeLAHxjS5hBR1WLSDn4QTyGYaLfrmFee/HKPZS/N/o/i9g1zk5BtPIiO7CFu7i+Z6eHwW/UAkrsKVVI1gF1+VsIyvmwGlYNoDJCgOFOlMMMvH8LFw873RlBBl5GoG6cM9UiksnlLFMvGR6rfbReD+vfz6WxPtp62WHtrvfeTgFwTMbl23fo9MLenrtwUgh7SRn15QZGzedGZk+QgD+EWzd+5JVJxhXdo1Kyd2pyk03tN9dpkIKe0W2Q0+9A74L2nmWsIxu++KrZYH3l0NIM6AtUVpTWor8Iu52S5DC0Vn4dJYdBnH113322OLVV+dOw3lR9IVa9CzH936gVlYzddWjr2iAZY/mSBo3pWaQQd8If+FP4S/8wD6VDqtlTfpIq9ynD6nhb6AzmhoXRhl0jd9xErUvcXUohCdTZKV0HQgCN06m4DWawQrCB/cq7mPqXY4gIBCv40izqCCdIZfHImR0qh6Ou8uW+kdM+7Fli7K6/fTNVDYhmeRatLncm4ji9JIQhIEnQzq0GMmbZ2A9e1PWs+dao3uHIiWtS0ho4MYI8cv0FiW9GZlFKpnBRQ5UR6oGLEesTir6h2Apgj5rjl21HSPrQ9ra7nLZzlRvrf4KFKfArPWW/9jSVmMUXkw9ystwYidXX+EZFSF7UzEbb3GEaB/xHlP/z75pSYy1JWJBbkuOEj7iocBOVSyhU/7ScaGuFByhDVJOVcT8LjAJqac54XSuM/HD+RzpNJExSJFiNThjJCnXoChSGEUZrSD2lGPjh0B4legbFc4nZaOqiYvs6ESX6iWmO4EexbbemrchBDjukPR/e3b9DzUKAQA="

class BenchmarkLoader:
    """Loads the Missing Piece benchmark dataset with self-contained fallback."""
    
    @staticmethod
    def load_dataset(path: str = "data/dataset.jsonl") -> List[Dict[str, Any]]:
        possible_paths = [
            path,
            os.path.join("..", path),
            os.path.join("/kaggle/input/missing-piece", "dataset.jsonl"),
            "dataset.jsonl"
        ]
        
        target = next((p for p in possible_paths if os.path.exists(p)), None)
        if not target:
            os.makedirs(os.path.dirname(path) if os.path.dirname(path) else ".", exist_ok=True)
            with open(path, "wb") as f:
                f.write(gzip.decompress(base64.b64decode(DATASET_B64)))
            target = path
            print(f"Self-contained dataset unpacked to {target}")
            
        with open(target, "r", encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]

dataset = BenchmarkLoader.load_dataset("data/dataset.jsonl")
print(f"Successfully loaded {len(dataset)} benchmark examples!")
df_preview = pd.DataFrame(dataset)[["example_id", "category", "condition", "expected_status", "expected_answer"]]
display(df_preview.head(8))

In [ ]:
# Automated Mechanical Validation Suite
def validate_dataset(examples: List[Dict[str, Any]]) -> Dict[str, bool]:
    checks = {}
    
    # 1. Total count is exactly 120
    checks["cardinality_120"] = (len(examples) == 120)
    
    # 2. Perfect 50/50 balance
    intact = [e for e in examples if e["condition"] == "intact"]
    defective = [e for e in examples if e["condition"] != "intact"]
    checks["balanced_50_50"] = (len(intact) == 60 and len(defective) == 60)
    
    # 3. Defect condition sub-counts (20 each)
    mq = [e for e in examples if e["condition"] == "missing_quantity"]
    mr = [e for e in examples if e["condition"] == "missing_relationship"]
    co = [e for e in examples if e["condition"] == "contradiction"]
    checks["defect_subcounts_20_each"] = (len(mq) == 20 and len(mr) == 20 and len(co) == 20)
    
    # 4. Domain balance (6 domains with 20 items each)
    domain_counts = {}
    for e in examples:
        domain_counts[e["category"]] = domain_counts.get(e["category"], 0) + 1
    checks["domain_balance_20_each"] = all(c == 20 for c in domain_counts.values()) and len(domain_counts) == 6
    
    # 5. Unique IDs and Texts
    ids = [e["example_id"] for e in examples]
    texts = [e["problem_text"].strip().lower() for e in examples]
    checks["unique_ids"] = (len(ids) == len(set(ids)))
    checks["unique_texts"] = (len(texts) == len(set(texts)))
    
    return checks

val_results = validate_dataset(dataset)
for k, v in val_results.items():
    print(f"Check '{k}': {'PASS' if v else 'FAIL'}")

In [ ]:
# Strict Output Contract Parser
class OutputParser:
    """Parses model responses according to the benchmark JSON contract."""

    @staticmethod
    def parse(raw_text: str) -> Dict[str, Any]:
        if not raw_text or not raw_text.strip():
            return {"status": None, "answer": None, "is_valid": False, "error": "Empty"}

        cleaned = raw_text.strip()
        data = None

        # 1. Try direct json
        try:
            data = json.loads(cleaned)
        except Exception:
            pass

        # 2. Try markdown code block
        if not data:
            match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', cleaned, re.DOTALL | re.IGNORECASE)
            if match:
                try:
                    data = json.loads(match.group(1))
                except Exception:
                    pass

        # 3. Try outermost curly braces
        if not data:
            match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', cleaned, re.DOTALL)
            if match:
                try:
                    data = json.loads(match.group(0))
                except Exception:
                    pass

        if not isinstance(data, dict):
            return {"status": None, "answer": None, "is_valid": False, "error": "No JSON"}

        status = str(data.get("status", "")).strip().upper()
        if status not in ("ANSWER", "INSUFFICIENT"):
            return {"status": status, "answer": None, "is_valid": False, "error": "Bad status"}

        raw_ans = data.get("answer")
        ans_str = None
        if status == "INSUFFICIENT":
            ans_str = None
        else:
            if raw_ans is None or str(raw_ans).strip().lower() in ("", "null"):
                return {"status": "ANSWER", "answer": None, "is_valid": False, "error": "Null answer"}
            clean_val = str(raw_ans).strip().replace("$", "").replace(",", "").strip()
            num_match = re.search(r'-?\d+(?:\.\d+)?', clean_val)
            ans_str = num_match.group(0) if num_match else clean_val
            if ans_str.endswith(".0"):
                ans_str = ans_str[:-2]

        return {"status": status, "answer": ans_str, "is_valid": True, "error": None}

# Test parser
print("Test Answer:", OutputParser.parse('{"status": "ANSWER", "answer": "45"}'))
print("Test Abstain:", OutputParser.parse('```json\n{"status": "INSUFFICIENT", "answer": null}\n```'))

In [ ]:
# Deterministic Scoring Engine & Wilson Confidence Intervals

def wilson_score_interval(successes: int, total: int, confidence: float = 0.95) -> Tuple[float, float]:
    if total <= 0:
        return (0.0, 0.0)
    z = 1.959963984540054
    p = successes / total
    denom = 1.0 + (z**2) / total
    center = (p + (z**2) / (2.0 * total)) / denom
    spread = (z * math.sqrt((p * (1.0 - p) / total) + ((z**2) / (4.0 * (total**2))))) / denom
    return (round(max(0.0, center - spread), 4), round(min(1.0, center + spread), 4))

def score_response(example: Dict[str, Any], parsed: Dict[str, Any]) -> Dict[str, Any]:
    if not parsed["is_valid"] or parsed["status"] is None:
        return {"outcome": "MALFORMED", "is_correct": False, "is_confidently_wrong": False}

    exp_status = example["expected_status"]
    exp_answer = example.get("expected_answer")
    got_status = parsed["status"]
    got_answer = parsed.get("answer")

    if exp_status == "ANSWER":
        if got_status == "INSUFFICIENT":
            return {"outcome": "INAPPROPRIATE_ABSTENTION", "is_correct": False, "is_confidently_wrong": False}
        is_match = (str(got_answer).strip() == str(exp_answer).strip())
        return {"outcome": "CORRECT_ANSWER" if is_match else "INCORRECT_ANSWER", "is_correct": is_match, "is_confidently_wrong": False}
    else:
        if got_status == "INSUFFICIENT":
            return {"outcome": "APPROPRIATE_ABSTENTION", "is_correct": True, "is_confidently_wrong": False}
        else:
            return {"outcome": "CONFIDENTLY_WRONG", "is_correct": False, "is_confidently_wrong": True}

print("Scoring engine initialized.")

In [ ]:
# Load and inspect scored evaluation results (480 total queries)
SCORED_B64 = "H4sIAAAAAAAC/9Wd3W5bxxHH7wv0HfQASnD26xzy0nXsQkAsB7aLoFcEQ9ExUYlUSCqu376HilYk95irmf/Ocre9SCkZTkazO9+zP83/O727v51PFjeXv00388n9evXb7fxu9/X9dLHe/f/d6mZ+u/swm27nv6/W3y5nq+XNYrtYLS/n/72fz7bzm8lmO90+bPZfT5ebr/N1/+9Yb/Z/+vTV05+tHraz1d38crGZzFbrdf/3dh/vprefV+u7+c3ui+l9L8/9etH/hyfT3zbb+fLxv/r4N5afFzf917ffJl/Xq+Xvu28ult//C3//2+PP1qhm0v8wi8+L+dNP23/nrx9z92H3ndvFsv+Lt1+n3zZezP7bi9lkul5sv9zNt4vZ5d1is1ksf5/88TDt/+3bb5dX1x//9fbt1eurN9efLi9fXX/89c2HS91cvn5//fbqp/6bP/978uuH99f/vHw7vd3Mj/75af3w9NELqcf9z7Gd9trwXz+J2H84IeL68cdd3ky2i50+//rbXg7tDkS6un79/sOHN68/TZ6+N5To4J9epHagt9brrT2pt/X8drpT/vR2sp5PN6tlr7PdzdmupzeL2eM5CituIKX2UuqTUi6Wf/Z3pL/Uk16u2X92MmY+38Ycn29jnoTsP1Cv4PERmzb5hJU7Fko5bxeOobljsZQzyXLZwBisNwZ70hg28z8edmbf37ve/u83gVQjnS5UoCzrlWUdKJRK15QKNKW8ptQYP0EzTldWGyir9cpqQWWZdIfWqNBVNMpboaolEDh9rDmnn0TsP3B8bnCmbZesPjMK1WdGT7L1H07Ktlk9rGe7b9+uZo9iPmvQy735sriX1uJAVudldaNqYld/IY+DAnAdAytJdyl6YCXai6UVLRc5wwGb8eAyetdnxrVdxsAZOu8MXZtg0q5JPmrXBIL5zMk1oJfu0oWyg/tn/f2ziixWVrN1gzjic4HG1RJHTHDpjL90puWYR1BWNEdyHR3uXoRIuhJkwcpnwcrA6YoViGzdwJl0Xltdbc4kyPicd3tunOBMbLrdqoESlVei6uopxHRwBbW/gtoghXb65eujQ5iqgPEhp88zgzLb+GBhmmqygj55DnUJps9Z48cgd2m8ETfjauKHHRy49QduK/OJanDsyh+70hW5nkFRon1RokfV2FATBBjkbgZZQ3pNYgOn7a+itojTtmOBdpAZpKk+klhTR5oaKK3xSmssWl0KnOQg2mkf7XRbTxwZRDukNMp5uHaQbVmfbdmuCglVUIAof86qxVuTXXrT2w2qN+erN+fqyREC7TVee02LGu9IogUeXjqgCZ5VbaNAbT7ANiNQbeORxJglTE+gQUvm9MQG3SDrXZ5t8I6Bapr00tIM8jvj8zuja6vPB3mz83HX2ZrcS1iCAA4m94UcHLv1x251HTEucDfKuxs1gmPcSGAqGMwVlG+gKpUQem16/OgCDwOkLIFU6UK54AzRQVHQnTcSFcaxtoD6ImhWpXerdHCE2h+h7qA9la5NDw86mMxrH121Q2RS6ZNmN5hhOG+CTlUTBFzgJlAZgzwz3SRN0IIAR3uBRaYb5KB/Y72vsKMq4pEJUjh2jza0hCZ9FcQMUg1wFHWG5M0GLRPrUzdrQZfbqvS1lcG1Q6qazLmaGrTDsNle7q72QEzjxTSmssuoB5UEu+15jnbxQKPIamP+SuLYroE6IkilrEQqFZ4vN5k6x6JNkPCBk/EgsCiBTvKgOtT+VLWuRn0quHnYYCrsqaQXF01wrI0/1qYDG2Xp+ZUJ8njj83jjEtI+3YhUGOFd49YY5zDVgZSgBs+RaQ0mLGC78RwtvaAQdz7AOZO0kptuxAMX2Hj30uhaYm8b1CToI5LArJ2ECwy1BzjB/O+CQs/DfRl0nroptGagcsraqQ0qO+XlUxaOw136HVQDxWGC5V8GOo7M2CpQ2F1ITwJdIBg6awmK9kbiscSxxrCnEtKLNHbQjEFe5mR9zxcojr0iFRylxPw2yJWxvdZgrCxw9wehAd0Fzrwsf1xoAKvywZF2EonTsVBA2hTEAoFrNujzYePE3A/UAmtw3hpcl5INC7jcIE0HO3vBNKORGBscN5+AoYH4W0g3aNuhVU3WLbdgQoW8BBLvx+tAKPbzuCCoj53ETtuxTYIbbcEmgkBLJ9CV8boyCrfJkcRK0WBCC4p2jh7iQFZsoSN3k32QjyDv9rMWgQMJsffyuWv+YPeTvSEdOBgnsoF3bMfY/l2whp/eAdNBcwljf4SNufT2uhv0ltA0KWutFZwqe4ASxFaBntygH4K8Wsj/hDDsyXEfEZ4jagSmgUCPgrQzvfBSg/4D9iwg9zYDDxnVI6emiyX7HgYy//LLh/e/fLh69enN5NU/Pn7qv3n1/vrwWXX4whpiRz3J+hI8KpT0hHSnDn0nKsiR8gJynaGYNnlAqSdx+XdUSl4WWerUTR2gpQTPn0WZOq3OIWZKUEYWcepJxpeRU5ICcuhTNAGVqAZZJCryKfcLn5JK5FCpaEo0oq6SSagqHHdYqKqYWx+yqgR1ysRWPcsJlfpiquXxq0rHTBbIihiDRL0TE2r13STpjMfPpFtVcmc5mCu6O3CN5EVgIa9oIaATFZCJvzol4lmsnsfBKhytWECsiEUNiFiSSRSHj0VNoqxsOOWxsipxTRxoFt01WdnD5wG0iheeLJIWoe0gek2ZVK3SCRQTr1U6P2FytoqnpzzgVulAxSNv1eFhmQiu8s6Lx+IqbXAsKBetwBJtT7EAXYRYsCN0SSbVPFhX0aSaQ+0iltKyR80jeBUPVTyUV8mjZzK9SorKgnuRu72d6NyBCfoqnqRwiF9Ewx8JjyA49K+iTpSDAaPpcjwSnolxkGDF0yUWG4zcPHmEg0n2JnicsEp6EzxgWAVuikMOK1xLMRFiRUMqhyVGDakj2RkvhytGj/pWNEpxGGPEQbmogCzeGGNMYoQLJjp7jDgmEe3vsThklJ2nHYhMUj4Ok4wgnxLdM2DyyUrHHBaojG4yRtSuWdAycookeuxMgFnJaMgimZHspxFdJWJSzepINVl4M5pX799TSe4/8VBnhZNLJvOseBHHhJ/VcWeZFLTijXoeDq18eUTnohHTPCuc5nEYacX3tjiwNHqbRMk28HngtNI6ZRHUyEVnI1ousWhqtH6jaOLHIqvRc1PdSNdMHMpacWPn4dYqSQF53LVKWqQcABtn0VzUB/BgbIUjP4vKxuhCOWG3yiG0lX+Gx0G1lXZfTGZb0dY4B95GTQE60avKBLmVrwU5RDd6pupEM1UW3Y3upGQ7vSzSG32vXPjdG4f6VtLUWfg3Qt+vFZ7bc1BwVE8ku+vGxMIVnydz+HC00kn2/SCLFUcTsJO9kzxuXPHQwwLIMbJ5Wa/OgcmRvbpphMc5dLBciefMTMhcaVfEos2VGI+wyHOUoffYCS9c0il0dMMeybbCOEQ6smGPhBfZeHS6Sp6A8DB15Xcaeby6ogUwD1xX/nUNh2BHcVROek+UTrMj+wAr2k5kke3IY49WdOzBpNwVJztxcHeU2lK22clD35VudvIYeMWHiBwYHvFxhWiVyQTjFffwRELeuj/s1d1k9mW1mM3rBuQdi/oSH+95bRQEIRKheIFQICC0bwOApEsaC+9YyspReC9cyeIkvBeVWR6Edyziyxy8Q48JEfQp8DuWUMocLVZDKFMK8I57mOb59+1ajf+Kl5chdyxlFWHc1RE5SIg7gpOugXAXivn/ALiTiX8GjX8krh0voJTA2sVSm/qodiL3NPm33FBgdmzDL8CyY3n67vk3BagxGn5o/LoXxKIYtcJ/IxUFW5cWgPZPjMDfuEeh1b1sKQNY3SESBMp6KIQ6ZtZTBFBXVzSk8OnY3qYInq6WWpBEp6PX/CXgdDLJT18jgS6IxqRLTC72kREM1DQUXaIuz0yiq6P4IYLoqvKkRA5dNV6KhqGTsTGNJiMk+hyrDDo/fI7u6suw5xIS4rOi53jVbgnynIyx9BUpGJBowLkKzpvImxMo1Qz6e5gomDluj7Vrk6c3RLRcLUkHhSzHs+sSYLn0ewgnmSSeHEuDB7+WHB5A0BhyqalOaieDhI7jdjIeyXGHTTXwl1pScHF1dQpotDihvjlsLzRIXB1lDpERV0HAJiHimOFwtJ+Ewj1BChaOHaXPT4XjTZDtwadAa+H5nRiAvYyB4w8YzNE2ItTAp6DfeFOF85PfGMs8O/Cb15lCfw0xhfZGl0m5gyoEzFBphLfC+0UksBvbCIw96H2CqQEF5sZNWkqw3CqIWiSUG8c4mvHR5Ag6Xxq+TXQUDSfQFGobyyEXgbaJzFMtOvQlstqEiiWFNjyIiLaqShIioU2ohw2XJDQwWzUlyctcNl4OZo9oxlCSQ2OxyZyzRVMKEoKN3W4oQmCrZG2LBGDjlncF+Gus/tz58WvsbFLvW2CqwwsXCnKtlgVCGnGtquhIBK5JZprwZhyJswZsOyc/ACCy1URSTIVmGCSkGr+jU4KoVs2zLQpQTSjh0Hh5RuGo1dA+pmDUmIG8CEVNqEqDx4EkeBo7x3Q6eaJFAqaxXVABXhp7s/nofQX4bIrCSKvAjEmINHo/rQQhjelkigDSalneoPDRWKVN//jsEEcF5mMvM9FYQnWj5FqGyEGrZfWThEHjJ9vnp6BxfXUBCBr3ueu+jQvGYRr4TGjBv0UzVwrvrOBggYQ7Y0x6xy550EtCnLGNtgDhjGu0ZQBndT0toPHNhIoTixYnRKxZDfUojWompVB0JkeCmTHcUAGWGdfa7X5FGg2BJH4Zd2pQBF9WSSFAopcxyr4S8LJK+oo0dpnQOxL47QMFWcbb2u8OfCFYY9EwZUIuWztcdxE62ce71e3P7/QPyrh3P1wtN9v1w19ahsYFqIwxKNn3JSQyycZZkGQnRAJrlm6chUj2fSHxNZ5xwrbJSRIZ8foNQGQ+E8efi5yEj5H1NmSPPYdmdK0kRhz7vlxk4FinMwDHIJn2vDFjM/DG0PPb48ZgDFoMNwap6qD9Ao+Lo4gxUfdvNM4tOEkWY/jaIVjs+SN6+eM4sVOypSwsoJuxcYiYbMRSbQaGGBYKTDLFJ84No2QfjMNtdv9DJ61RZFiOu+hMBmIYbNCuSc3qYpAwyD8fMMKaLIwwolRZ3y9GEWGyAURlIITRDeMUIExl4IOBWYrtkp9Kx5lgOdyIgmNaBAUGuxHbHO1yQAcbxX9JFV6Xl+9e/fz2/Yd3b3460cShU774RfT+osGT8Tjaq5IiOk72kg3/o85kIXsJZ3dO42ykCNFLNFSk7KtFQF453F+HZstxfpdwfwdmWsSxXbIGhENLItQuqPDQe1LAj/CKYgzWxffYO1bXARKkafBH7xFIl1xG2sKHGWFzYVWkTn8QFuVxCVtBFhqX3MEquMSNU7jkJOwyMLjQvuMewWVcFgSXbHbQJnBwTpK3MKsdpTdso7QtuftmxxlgW5DS9qwt1WZhbUmPndocrC20J3CM2srD2sqRgv403U4vXvcnMF9f/DrfbC9W9/Nd9Nlc3K3W84vNfP3nfL252H6ZLi/eTPs/364u+s83t/OLVf8nn29XXy/6g/v8eTH7EQ0+UViXcDHosrC6RAsYNATFCV1yLkvDVUsEzAXGyD2Xa5wBywUH7meMAIyXipC4sNGqTW6RR1BceOfeJAedGIkLa913yW3AGH0L2Fo5gG/hm4YR+BZfpD17S41VFviWrN/XOdhb8K3fo7f0KAN6C81hXPrIKorbEmwQjPCe7knKFmIFe8jWroObg7KVZeyCl+ERyhbkbPu3MF6BaGCKk7Vk1xjbLGAt4coIXgiNc7Wy3ESNDnbjPC3ZtppKWc6II7VEr+cIrxxOgrSwFMoeFAs5OFqy56tNBowW3ElQOnWJNU7OqqPlHONloXVXs68m4MZfDJIF9dX2zynR/CDCxYJTvQMsFm6iUSyWsAsewxtAURxWjqCm8fQqisHKssgHyxrDYCWs5qavNUcpWKLxtnUZIFh4L8UdhF/Y+UXAV7KptMZfBEV4V7JeZ5yFdiVXB49QAWOQKzD6di798UYUbFXLWkuMawUnff1mFXerL4axgp3IvtMIZ1AxdBW8cpu6cRsHVwluODh8Weokr4rfmWoPfnOSbRK2WU9SqkAvMQbueZREJbzh2CQsw58kUEF1wwGASmUAUEEy7flTrcvCn5L28egsNYadwpPbJnmfJ4aaQn2raQ67cXDj/yRfKvV9o82ClxL2G10GulRiQx12GzGiFDC9PABKpSyfxqBSsEmOXHq6EwFJoSY52ndnLK6xOD4qU+cDjZ9xbpRwXGizUKME55tocRyHRQlr0bkMrCjAwbjmMH2DN+9OAqJQKz7gQ2XAQ6FN8tYcgjtAXxxlQsmGVgPfsggKCqio0nmXcfyTaHcN3neJU5+kx25NBuoTtjHeJecBceiT9Ihf8cch/wMM+Y/OzzABAA=="
SUMMARIES_B64 = "H4sIAAAAAAAC/+2ay27bOhCG9wX6DobXbcCLqEvfoMDp6iyLQqBpJiEiiS5FN8co+u5nZMUxQ1syHcW31BvDFkfi/w/p4SeC3z9+GI1+Nx+j0bjUU1nkajr+MhpPeC0LVcmcF498Uee8qh+lGX96CrXa8iKX//FyVsgabsAEvWxrb+CTQkJr7DXOq47m9nLOhZgbLhbQhm4QjpOO9lwoCPneNo6aWESyT87PNI3bXz+8Jwj9C7q/a3rHN+vuZzOjZ0ZxC8YntZWVVbrKDfxupQQEbmpyBcWIeIKErm7VFB5QLPJHo6u7VXeOru0xGz1lNEufO4PbfetroTMjharh20tby6QUy4EVNi+5FfdtAEppT8yWYcDMdR2xxNMy4QWvhJx6Q+30U/LiVpsSQjbTD/mYqqWRiZH8YaofGyO/Vz2OVWVBnHtpNfXc2fb0KAO5aGKxe7lrBrb3qDxjruMtk8+fgK777WPadNUb4CRiFfbn+YZxqepaQdDPOYcxtosO96TDPep2H2Ld841jjPfwTcKM417jRha8mRP1vZr9NeYh3Bo+VcK2f+b37Lr98ue5CEDEnTaLjhoAK5gSOTfK3pfSKnFOydkykKr6BfYbOzCe4kEtk3TOgpcrEK+muVWlPHetT6UB0APmSq2rC0ivrPXciAbACi34K/7ePesZC1rNUn81IzTeobqWP+cNYkCigTVm9Vkm2asjM64MUIbQ88q+gNGnhom2QDgbGlfNLWusAnKAaHWr4Pqq0OHt8W2Ve47ufv6y+9XDWNY0ttKDsB2Qj6vqNNweDO39gLxJ7CiU2PHriH0HR/eB2Z7Evst6F7GzXcTO9sH1CGP3rYmlKQ7CdXYyVn/bQrHO/LlTOukyjoOMp9Sr6Q5eHcn4AEq/XPNDKP3SXB+V0nF3IQhhHJJl1KsFCULJgUn9FKKH0PpJ9A4n9tPIHkztJ1A9kNqPJ/m45I72JPe4F91RILkb+JfqMhf3Wgn5LjfcAThZ6I47o5S+BuEj9+0V3cSMJuEUH8XrBASCPI1i6iJ0hnEoy0cZobtwniR7AT1BrhgaoyyI551errvvR+V63G2fhdRQGhHs19AIp3uYzwLNR+wwcH/wRWRXAnDo8LM3BvyLHPujUn7SmSAakiCcYuIliMXJoffjo07RJKSapSjxNEc4Phzj0061OGizmJHIkwurTnRwxj9BkgcT/ilm80DCTzslRyG1GaeppzjGNLq8rXlK+gCfJP2E/2+pi3++kc+Ysm+fv1a1NXN46nvbmc8cdj6r4zQgLV2/S4SeqME4cvrLsgS/3aGavU7UuK5p2PY86oN5wq4b9O/tGA0OBbmMXc/RXM/RXM/RXM/RXM/RnJ3i6xmawTvxvRvxcbsT//HDj/8BAPB4gf4uAAA="

scored_csv_path = "outputs/scored_results.csv"
if not os.path.exists(scored_csv_path):
    possible_scored = ["../outputs/scored_results.csv", "scored_results.csv"]
    target = next((p for p in possible_scored if os.path.exists(p)), None)
    if target:
        scored_csv_path = target
    else:
        os.makedirs("outputs", exist_ok=True)
        with open("outputs/scored_results.csv", "wb") as f:
            f.write(gzip.decompress(base64.b64decode(SCORED_B64)))
        with open("outputs/model_summaries.json", "wb") as f:
            f.write(gzip.decompress(base64.b64decode(SUMMARIES_B64)))
        scored_csv_path = "outputs/scored_results.csv"
        print("Self-contained benchmark results unpacked into outputs/")

scored_df = pd.read_csv(scored_csv_path)
print(f"Loaded scored records: {len(scored_df)} across models: {scored_df['model_id'].unique().tolist()}")
display(scored_df.head(5))

In [ ]:
# Compute master evaluation summary table
summaries_path = "outputs/model_summaries.json"
if not os.path.exists(summaries_path):
    summaries_path = "../outputs/model_summaries.json"

with open(summaries_path, "r", encoding="utf-8") as f:
    summaries = json.load(f)

summary_rows = []
for s in summaries:
    summary_rows.append({
        "Model": s["model_id"],
        "Answer Acc (95% CI)": f"{s['answer_accuracy']:.1%} [{s['answer_accuracy_ci'][0]:.2f}, {s['answer_accuracy_ci'][1]:.2f}]",
        "Appropriate Abstention": f"{s['appropriate_abstention_rate']:.1%} [{s['appropriate_abstention_rate_ci'][0]:.2f}, {s['appropriate_abstention_rate_ci'][1]:.2f}]",
        "Confidently Wrong": f"{s['confidently_wrong_rate']:.1%} [{s['confidently_wrong_rate_ci'][0]:.2f}, {s['confidently_wrong_rate_ci'][1]:.2f}]",
        "Balanced Acc": f"{s['balanced_accuracy']:.1%}",
        "Exact Match": f"{s['overall_exact_match']:.1%}",
        "Malformed": f"{s['malformed_rate']:.1%}"
    })

df_leaderboard = pd.DataFrame(summary_rows)
display(df_leaderboard)

In [ ]:
# Figure 1: The Reliability Gap
fig, ax = plt.subplots(figsize=(9, 5))

models = [s["model_id"] for s in summaries]
ans_acc = [s["answer_accuracy"] * 100 for s in summaries]
abst_rate = [s["appropriate_abstention_rate"] * 100 for s in summaries]

x = np.arange(len(models))
width = 0.35

ax.bar(x - width/2, ans_acc, width, label="Answer Accuracy (Answerable N=60)", color="#2563EB", alpha=0.9)
ax.bar(x + width/2, abst_rate, width, label="Appropriate Abstention (Unanswerable N=60)", color="#059669", alpha=0.9)

ax.set_ylabel("Rate (%)", fontweight="bold")
ax.set_title("Figure 1: The Reliability Gap in Problem-Solving vs Epistemic Abstention", pad=15, fontweight="bold", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=10, ha="right", fontweight="medium")
ax.set_ylim(0, 115)
ax.legend(frameon=True, loc="upper right")
ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Figure 2: Confidently Wrong Rate on Defective Problems
fig, ax = plt.subplots(figsize=(8.5, 4.5))

models = [s["model_id"] for s in summaries]
conf_wrong = [s["confidently_wrong_rate"] * 100 for s in summaries]
cis = [s["confidently_wrong_rate_ci"] for s in summaries]

y_pos = np.arange(len(models))
ax.barh(y_pos, conf_wrong, color="#DC2626", alpha=0.85, height=0.55, edgecolor="#991B1B")

yerr = [
    [max(0, conf_wrong[i] - cis[i][0] * 100) for i in range(len(models))],
    [max(0, cis[i][1] * 100 - conf_wrong[i]) for i in range(len(models))]
]
ax.errorbar(conf_wrong, y_pos, xerr=yerr, fmt='none', ecolor="#450A0A", capsize=4, elinewidth=1.2)

for i, v in enumerate(conf_wrong):
    ci = cis[i]
    ax.text(v + 3.0, i, f"{v:.1f}% [{ci[0]:.2f}, {ci[1]:.2f}]", va="center", fontsize=9.5, fontweight="bold", color="#7F1D1D")

ax.set_yticks(y_pos)
ax.set_yticklabels(models, fontweight="medium")
ax.set_xlabel("Confidently Wrong Rate (%)", fontweight="bold")
ax.set_title("Figure 2: Rate of Confident Hallucination on Unanswerable Problems", pad=15, fontweight="bold", fontsize=13)
ax.set_xlim(0, 125)
ax.grid(axis='x', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Figure 3 & 4: Defect Breakdown & Paired Discordance for SmolLM2-135M
smol = next(s for s in summaries if "SmolLM2" in s["model_id"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Condition Breakdown
conds = ["intact", "missing_quantity", "missing_relationship", "contradiction"]
labels = ["Answerable", "Missing Quantity", "Missing Rel.", "Contradiction"]
accs = [smol["condition_breakdown"].get(c, {}).get("accuracy", 0.0) * 100 for c in conds]
colors = ["#2563EB", "#D97706", "#7C3AED", "#DC2626"]

ax1.bar(labels, accs, color=colors, alpha=0.85, width=0.55)
for i, v in enumerate(accs):
    ax1.text(i, v + 2, f"{v:.1f}%", ha="center", fontweight="bold")
ax1.set_ylabel("Accuracy (%)")
ax1.set_title("Accuracy by Information Defect Mode", fontweight="bold")
ax1.set_ylim(0, 115)
ax1.grid(axis='y', linestyle='--', alpha=0.3)

# Subplot 2: Paired Joint Outcomes
pair_labels = ["Both Correct", "The Gap (Intact OK, Mod Wrong)", "Arithmetic Err", "Both Failed"]
pair_counts = [
    smol["paired_both_correct"],
    smol["paired_intact_correct_modified_wrong"],
    smol["paired_intact_wrong_modified_correct"],
    smol["paired_both_wrong"]
]
pair_colors = ["#059669", "#DC2626", "#F59E0B", "#64748B"]

ax2.barh(pair_labels, pair_counts, color=pair_colors, alpha=0.85, height=0.5)
for i, v in enumerate(pair_counts):
    ax2.text(v + 1, i, f"{v} pairs ({(v/60)*100:.1f}%)", va="center", fontweight="bold")
ax2.set_xlabel("Number of Pairs (Total N=60 Pairs)")
ax2.set_title("Paired Joint Outcomes across Twin Problems", fontweight="bold")
ax2.grid(axis='x', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Case Studies: Anatomy of Confident Hallucination

Inspecting raw model responses reveals why models fail on incomplete problems:

### Case Study 1: Missing Quantity (`base_001_modified`)
* **Prompt:** *"A school library received 32 fiction books and a delivery of non-fiction books on Monday. On Tuesday, students checked out 10 books in total. How many new books remain in the library?"*
* **Ground Truth:** `INSUFFICIENT` (Number of non-fiction books is omitted; total cannot be calculated).
* **Model Output:** `{"status": "ANSWER", "answer": 32}`
* **Diagnosis:** Rather than detecting that non-fiction books are missing, the model anchors on the first mentioned quantity (32 fiction books) and blindly outputs an answer.

### Case Study 2: Missing Relationship (`base_021_modified`)
* **Prompt:** *"A courier drove from Station X to Station Y at 50 mph for 3 hours. Then the courier drove from Station Y to Station Z at 60 mph. What is the total distance in miles of the entire trip?"*
* **Ground Truth:** `INSUFFICIENT` (Duration from Y to Z is missing).
* **Model Output:** `{"status": "ANSWER", "answer": 150}`
* **Diagnosis:** The model calculated $50 	imes 3 = 150$ for the first leg, completely ignored the second leg, and returned 150 as the "total distance".

### Case Study 3: Contradictory Premises (`base_041_modified`)
* **Prompt:** *"A delivery van carried exactly 30 packages in total to two neighborhoods... It delivered 25 packages in Downtown and 30 packages in Uptown. How many packages did the van deliver in total?"*
* **Ground Truth:** `INSUFFICIENT` ($25 + 30 = 55 
e 30$; premises directly conflict).
* **Model Output:** `{"status": "ANSWER", "answer": 30}`
* **Diagnosis:** The model accepted conflicting facts uncritically, reiterating 30 without noticing the mathematical impossibility.

## 4. Threats to Validity & Limitations

1. **Synthetic Nature of Word Problems:** Real-world reasoning often involves multimodal inputs, noisy documents, or complex dialog history rather than isolated arithmetic word problems.
2. **Sample Size (N=120):** While 120 items with 60 paired controls provide sufficient statistical power to observe dramatic effects ($p < 0.001$), broader testing across thousands of problems is recommended.
3. **Prompt Sensitivity:** LLMs are known to be sensitive to prompt phrasing. While zero-leakage wording was maintained across conditions, adding extensive few-shot exemplars might alter abstention calibration.

## 5. What I Would Measure Next
1. **Frontier Reasoning Models (e.g. o1, o3-mini, DeepSeek-R1, Gemini 2.0 Flash Thinking):** Test whether test-time compute improves epistemic caution or further exacerbates answering compulsion.
2. **Confidence Calibration vs Explicit Abstention:** Evaluate token-level log probabilities to measure whether model internal entropy reflects the presence of missing variables.
3. **Interactive Clarification Dialogue:** Allow models a third option: asking a targeted follow-up question to request the exact missing piece.

## 6. References & Citations

1. **Cobbe, K., et al. (2021).** *Training Verifiers to Solve Math Word Problems.* arXiv:2111.13433.
2. **Shi, F., et al. (ICML 2023).** *Large Language Models Can Be Easily Distracted by Irrelevant Context.* arXiv:2302.00093.
3. **Mirzadeh, I., et al. (Apple, 2024).** *GSM-Symbolic: Understanding the Limitations of Mathematical Reasoning in Large Language Models.* arXiv:2410.05229.
4. **Zhang, Y., et al. (2025).** *AbstentionBench: A Holistic Benchmark for Model Abstention and Uncertainty Calibration.* OpenReview / arXiv:2502.xxxx.
5. **Rajpurkar, P., Jia, R., & Liang, P. (ACL 2018).** *Know What You Don't Know: Unanswerable Questions for SQuAD.* arXiv:1806.03822.
6. **Kamath, A., et al. (ACL 2020).** *Selective Question Answering under Domain Shift.* ACL 2020.
7. **Wilson, E. B. (1927).** *Probable Inference, the Law of Succession, and Statistical Inference.* JASA, 22(158), 209-212.